**Так как тексты в сфере образования, то подойдут и текста образовательных курсов (Stepik, 2035 и прочих)**

**Чтобы не мучаться с "поступи онлайн", "вузопедия" и прочими**

# Stepik

In [10]:
#для запросов
import requests

In [13]:
#собираем теги курсов, в названиях которых есть ключевые слова
all_tags = {}
auth_url = "https://stepik.org/api/tags"
for page in range(1, 50):
    response = requests.get(f"{auth_url}?page={page}&language=ru")
    if response.status_code != 200:
        break
            
    data = response.json()
    tags = data.get('tags', [])
        
    if not tags:
        break
            
    for tag in tags:
        all_tags[tag['id']] = tag['title']
            
    #есть ли следующая страница
    if not data.get('meta', {}).get('has_next'):
        break
            
delete = []
for tid, title in all_tags.items():
    if any(word in title.lower() for word in ['физик', 'матем', 'инжен', 'хим', 'гео', 'электр', 'инфо']):
        print(f"{tid} {title}")
    else:
        delete.append(tid)

for tid in delete:
    del all_tags[tid]

1 Информационные технологии
100 Информационная безопасность и кибербезопасность
101 Электроника и аппаратное обеспечение
116 Физика
123 Химия
126 Математика и статистика
129 Математический анализ
130 Геометрия и топология
132 Дискретная математика
133 Вычислительная математика
134 Прикладная математика
135 Математическое моделирование
137 Математическая логика
151 География
218 Маркетинг электронной торговли
230 Электронный документооборот
234 Электронная коммерция
305 Математика
306 Информатика
311 Химия
312 Физика
349 Инженерное дело
353 Электротехника
357 Инженерное проектирование и моделирование
375 Биоинформатика
383 География
388 Теория информации


In [15]:
#дополнительно почистим
delete = [218, 375]
for tid in delete:
    del all_tags[tid]
#вывод остатка
print(all_tags)

{1: 'Информационные технологии', 100: 'Информационная безопасность и кибербезопасность', 101: 'Электроника и аппаратное обеспечение', 116: 'Физика', 123: 'Химия', 126: 'Математика и статистика', 129: 'Математический анализ', 130: 'Геометрия и топология', 132: 'Дискретная математика', 133: 'Вычислительная математика', 134: 'Прикладная математика', 135: 'Математическое моделирование', 137: 'Математическая логика', 151: 'География', 230: 'Электронный документооборот', 234: 'Электронная коммерция', 305: 'Математика', 306: 'Информатика', 311: 'Химия', 312: 'Физика', 349: 'Инженерное дело', 353: 'Электротехника', 357: 'Инженерное проектирование и моделирование', 383: 'География', 388: 'Теория информации'}


In [27]:
import re, json
#оставляем только текст из html
def clean_html(raw_html):
    return re.sub(re.compile('<.*?>'), '', raw_html)

In [29]:
import pandas as pd
rows = []
for tid, tag_name in all_tags.items():
    #на одной странице максимум 20 курсов
    for page in range(1, 3):
        params = {
                'tag': tid,
                'page': page,
                'language': 'ru',
                'is_popular': 'true' #берем проверенные
        }
        try:
            response = requests.get("https://stepik.org/api/courses", params=params)
            if response.status_code != 200:
                break
            data = response.json()
            courses = data.get('courses', [])
            if not courses:
                break
                
            for course in courses:
                rows.append({
                    "category": tag_name,
                    "title": course.get("title"),
                    "summary": clean_html(course.get("summary")),
                    "description": clean_html(course.get("description")),
                    "target_audience": clean_html(course.get("target_audience"))
                    })
                
                
        except Exception as e:
            print(tid)
            break




In [30]:
df_cources = pd.DataFrame(rows)

In [31]:
df_cources

,category,title,summary,description,target_audience
0,Информационные технологии,Тестирование ЕГЭ/ОГЭ,"Этот курс создан специально для тех, кто сдает...",Ключевая характеристика:\nЭто не теоретический...,"Учащиеся 9 - 11 классов, которые готовятся к с..."
1,Информационные технологии,API автоматизация тестирования на Python,"API тестирование — навык, который отделяет джу...",Содержание курса\n\n\n\tИзучение Python\n\n\t\...,"- Для новичков: Для тех, кто начинает свой пут..."
2,Информационные технологии,TS: Магазинчик на React'e [драфт],ЭТОТ КУРС ПОЯВИТСЯ ТОЛЬКО В СЕРЕДИНЕ МАЯ 🚀 Одн...,Это курс о frontend-разработке на практике. Ba...,"Для начинающих frontend-разработчиков, но не с..."
3,Информационные технологии,Публикация сайта в интернете,Научитесь с нуля публиковать сайт в интернете:...,Этот курс показывает полный путь публикации са...,"Новички, которые хотят понять, как работает ин..."
4,Информационные технологии,Инженер по тестированию: путь к веб-автоматиза...,Освойте профессию инженера по тестированию ПО ...,Инженер по тестированию: путь к веб-автоматиза...,"- Новички, которые только начинают погружение ..."
...,...,...,...,...,...
829,Теория информации,Введение в теорию защиты информации,"Курс МДК.03.01 Технические методы и средства, ...","Курс МДК.03.01 ""Технические методы и средства,...",студенты обучающиеся на техника по защите инфо...
830,Теория информации,Основы теории информации,Теория информации (ТИ) – это прикладная наука...,Данный онлайн-курс представляет собой введени...,Студенты СПО и все интересующиеся
831,Теория информации,Квантовые вычисления,В рамках данного курса слушатели познакомятся ...,"О квантовых вычислениях много пишут и говорят,...","Мы ждем на курсе математиков и программистов, ..."
832,Теория информации,"Измерение, кодирование и обработка информации",Курс представляет собой собрание лучших практи...,Основная цель курса - углубление знаний и подг...,"все желающие, 10-классники."


In [36]:
#почистим от школьников
except_words = ['ЕГЭ', 'ОГЭ', 'школ']
pattern = '|'.join(except_words)
df_cources = df_cources[
    ~df_cources['title'].str.contains(pattern, case=False, na=False) & 
    ~df_cources['summary'].str.contains(pattern, case=False, na=False)
]
df_cources

,category,title,summary,description,target_audience
1,Информационные технологии,API автоматизация тестирования на Python,"API тестирование — навык, который отделяет джу...",Содержание курса\n\n\n\tИзучение Python\n\n\t\...,"- Для новичков: Для тех, кто начинает свой пут..."
2,Информационные технологии,TS: Магазинчик на React'e [драфт],ЭТОТ КУРС ПОЯВИТСЯ ТОЛЬКО В СЕРЕДИНЕ МАЯ 🚀 Одн...,Это курс о frontend-разработке на практике. Ba...,"Для начинающих frontend-разработчиков, но не с..."
3,Информационные технологии,Публикация сайта в интернете,Научитесь с нуля публиковать сайт в интернете:...,Этот курс показывает полный путь публикации са...,"Новички, которые хотят понять, как работает ин..."
4,Информационные технологии,Инженер по тестированию: путь к веб-автоматиза...,Освойте профессию инженера по тестированию ПО ...,Инженер по тестированию: путь к веб-автоматиза...,"- Новички, которые только начинают погружение ..."
5,Информационные технологии,ClickHouse: продвинутый уровень,🚀 ClickHouse — самая популярная в мире аналити...,Добро пожаловать на курс!\n\nЗадать вопросы пе...,"Аналитики данных (DA), инженеры данных (DE). Х..."
...,...,...,...,...,...
828,Теория информации,Алгоритмы на Python,Алгоритмы и структуры данных на языке Python. ...,Серьезный курс для будущих профессионалов.\n\n...,Программистам на Python\nНаучитесь писать эффе...
829,Теория информации,Введение в теорию защиты информации,"Курс МДК.03.01 Технические методы и средства, ...","Курс МДК.03.01 ""Технические методы и средства,...",студенты обучающиеся на техника по защите инфо...
830,Теория информации,Основы теории информации,Теория информации (ТИ) – это прикладная наука...,Данный онлайн-курс представляет собой введени...,Студенты СПО и все интересующиеся
831,Теория информации,Квантовые вычисления,В рамках данного курса слушатели познакомятся ...,"О квантовых вычислениях много пишут и говорят,...","Мы ждем на курсе математиков и программистов, ..."


# 2035

In [49]:
#вытаскиваем id всех курсов
#чистить не будем, так как сайт с техническими доп курсами (школьных и чисто гум курсов нет)
response = requests.get("https://cat.2035.university/rall/courses/?display_hidden=0&_=1775421511028")
courses = response.json()['courses_ids']

In [70]:
def parse_2035(course_id):
    try:
        url = f"https://cat.2035.university/rall/course/{course_id}/"
        response = requests.get(url, timeout=2)
        response.encoding = 'utf-8'
        if response.status_code != 200:
            return None
        
        soup = BeautifulSoup(response.text, 'html.parser')
        #название курса (возвращаем none, чтобы не добавить 404)
        title_el = soup.find('h1', class_='course-title')
        if not title_el:
            return None
        title = title_el.get_text(strip=True)
    
        #категория (может быть не указана)
        category_el = soup.find('span', class_='course-tools-badge-active')
        category = category_el.get_text(strip=True) if category_el else "n/a"
    
        #описание (весь основной контент)
        desc_parts = []
        main_content = soup.find('div', class_='course-content--main--body')
        if main_content:
            desc_parts.append(main_content.get_text(strip=True))
        
        sub_headers = soup.find_all('h5', class_='course-content-block-sub-header')
        for sh in sub_headers:
            desc_parts.append(sh.get_text(strip=True))
        
        description = " ".join(desc_parts) if desc_parts else "n/a"
    
        #summary (сборник навыков)
        summary_el = soup.find('div', class_='competence-contents')
        summary = summary_el.get_text(strip=True) if summary_el else "n/a"
        
        #аудитория (почему бы и не добавить)
        audience_els = soup.find_all('p', class_='prof-standard-name')
        if audience_els:
            audience = "; ".join([" ".join(el.get_text().split()) for el in audience_els])
        else:
            audience = "n/a"
    
        return {
            "category": category,
            "title": title,
            "summary": summary,
            "description": description,
            "target_audience" : audience
        }
    except Exception as e:
        print(course_id)

#проверяем на курсе
print(parse_2035(18697))

{'category': 'Программирование и создание ИТ-продуктов', 'title': 'Основы тестирования ПО', 'summary': 'Знать:●\tНормативно-технические материалы по вопросам испытания и тестирования ПО;●\tОсновные термины и сокращения, используемые в технической документации и принятые в организации;●\tОсновы работы в операционной системе, в которой производится тестирование, на уровне, необходимом для тестирования ПО соответствующего типа;●\tОсновы теории алгоритмов и дискретной математики в объеме полученного профессионального образования;●\tСинтаксис языка программирования тестируемого ПО, особенности программирования на этом языке, стандартные библиотеки языка программирования;●\tЖизненный цикл дефекта ПО;●\tПравила оформления технической документации;●\tОсновные термины и сокращения, используемые в технической документации и принятые в организации;●\tПринципы работы в системе контроля дефектов;●\tОсновные инструментальные средства организации работы в команде;●\tОсновные понятия о качестве ПО;●\t

In [71]:
#очень длинный вывод
counter = 0
for i in courses:
    res = parse_2035(i)
    if res:
        rows.append(res)
        counter += 1
        print(counter)
    if counter == 400:
        break

1
2
3
4
5
6
7
8
19466
9
11369
10
11
12
13
14
15
16
17
18
11111
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
11291
49
50
51
52
53
18704
54
55
56
57
16944
58
59
60
11193
61
18694
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
23843
18795
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
11514
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
26

In [72]:
df_2035 = pd.DataFrame(rows)

In [73]:
df_2035

,category,title,summary,description,target_audience
0,Программирование и создание ИТ-продуктов,Профессиональный разработчик 1С,Знать:•\tобщую концепцию и структуру технологи...,ОписаниеПрограмма «Профессиональный разработч...,Информационные системы и технологии
1,Управление цифровой трансформацией,Менеджер проектов,Знать:• Аналитика как этап производства продук...,"Описание""В течение пяти рабочих дней с момента...",Менеджмент проектов в области информационных т...
2,Цифровой маркетинг и медиа,Интернет-маркетолог,"Знать:Анализ здоровья бизнеса, формулирование ...",ОписаниеВ течение пяти рабочих дней с момента ...,Специалист по интернет-маркетингу
3,Цифровой маркетинг и медиа,Руководитель интернет продаж - Управляющий инт...,Знать:• Разбираться в видах стратегий бизнеса ...,ОписаниеВ процессе обучения студенты учатся уп...,Менеджер по продажам информационно- коммуникац...
4,Управление цифровой трансформацией,Создание онлайн-курса. С нуля до первого запуска,Знать:Методики составления образовательного ...,"ОписаниеЦелью образовательной программы ""Созда...",Педагогическая деятельность в дополнительном о...
...,...,...,...,...,...
761,n/a,Технологии смешанного обучения (blended learning),n/a,ОписаниеВ рамках курса слешатели познакомятся ...,n/a
762,n/a,Основы программирования в LabVIEW,n/a,"ОписаниеКурс ""Основы программирования в LabVIE...",n/a
763,n/a,Промышленный дизайн и 3D-моделирование,n/a,ОписаниеВ современном мире дизайн охватывает п...,n/a
764,n/a,Разработка Android-приложений для мобильных ус...,n/a,ОписаниеДанная программа направлена на формиро...,n/a


# Объединение + экспорт

In [75]:
out_df = pd.concat([df_cources, df_2035])

In [76]:
out_df

,category,title,summary,description,target_audience
1,Информационные технологии,API автоматизация тестирования на Python,"API тестирование — навык, который отделяет джу...",Содержание курса\n\n\n\tИзучение Python\n\n\t\...,"- Для новичков: Для тех, кто начинает свой пут..."
2,Информационные технологии,TS: Магазинчик на React'e [драфт],ЭТОТ КУРС ПОЯВИТСЯ ТОЛЬКО В СЕРЕДИНЕ МАЯ 🚀 Одн...,Это курс о frontend-разработке на практике. Ba...,"Для начинающих frontend-разработчиков, но не с..."
3,Информационные технологии,Публикация сайта в интернете,Научитесь с нуля публиковать сайт в интернете:...,Этот курс показывает полный путь публикации са...,"Новички, которые хотят понять, как работает ин..."
4,Информационные технологии,Инженер по тестированию: путь к веб-автоматиза...,Освойте профессию инженера по тестированию ПО ...,Инженер по тестированию: путь к веб-автоматиза...,"- Новички, которые только начинают погружение ..."
5,Информационные технологии,ClickHouse: продвинутый уровень,🚀 ClickHouse — самая популярная в мире аналити...,Добро пожаловать на курс!\n\nЗадать вопросы пе...,"Аналитики данных (DA), инженеры данных (DE). Х..."
...,...,...,...,...,...
761,n/a,Технологии смешанного обучения (blended learning),n/a,ОписаниеВ рамках курса слешатели познакомятся ...,n/a
762,n/a,Основы программирования в LabVIEW,n/a,"ОписаниеКурс ""Основы программирования в LabVIE...",n/a
763,n/a,Промышленный дизайн и 3D-моделирование,n/a,ОписаниеВ современном мире дизайн охватывает п...,n/a
764,n/a,Разработка Android-приложений для мобильных ус...,n/a,ОписаниеДанная программа направлена на формиро...,n/a


In [78]:
#сохраняем для дальнейшей работы
out_df.to_csv('nlp3.csv', index=False)